# Create App Connection

Creates (or replaces) the Unity Catalog HTTP connections to the
Hi Genie Orchestrator app:

1. **REST connection** (`base_path: /api`) — Used by UC functions via `http_request(conn => ...)`
2. **MCP connection** (`base_path: /mcp`) — Single connection backing all 5 MCP services.
   Tool scoping is handled by `include_tool_selectors` on each service, not by routing.

**Parameters** (passed via job `base_parameters`):
| Parameter | Description |
|-----------|-------------|
| `connection_name` | REST API connection name (base_path: /api) |
| `mcp_connection_name` | Single MCP connection name (base_path: /mcp, all 39 tools) |
| `catalog` | UC catalog for grants |
| `schema` | UC schema for grants |
| `app_host` | App hostname (no protocol) |
| `secret_scope_name` | Secret scope containing SPN credentials |
| `client_id_dbs_key` | Key name for OAuth client_id in the scope |
| `client_secret_dbs_key` | Key name for OAuth client_secret in the scope |
| `workspace_host` | Workspace hostname (no protocol) for token endpoint |
| `workers_group_name` | Group receiving USE CONNECTION + catalog/schema grants |

> **Two connections, one app:**
> - REST connection (`/api`) — UC functions
> - MCP connection (`/mcp`) — all 39 tools; persona scoping via UC service selectors
> Both use the same SPN credentials and app host.
>
> **Idempotency:** Uses `CREATE CONNECTION IF NOT EXISTS` — creates on first
> deploy, no-op if already exists. The reset (for credential rotation) is handled
> by a separate task gated by the `drop_connection` job parameter.

In [0]:
-- REST API connection: UC functions use http_request(conn => ..., path => '/v1/...')
-- base_path /api is prepended, so final URL = https://<app>/api/v1/...
CREATE CONNECTION IF NOT EXISTS ${connection_name}
TYPE HTTP
OPTIONS (
  host 'https://${app_host}',
  base_path '/api',
  client_id secret('${secret_scope_name}', '${client_id_dbs_key}'),
  client_secret secret('${secret_scope_name}', '${client_secret_dbs_key}'),
  oauth_scope 'all-apis',
  token_endpoint 'https://${workspace_host}/oidc/v1/token'
)

In [0]:
-- Single MCP connection: exposes all 39 tools at /mcp.
-- Persona scoping is handled by include_tool_selectors on each UC MCP Service.
CREATE CONNECTION IF NOT EXISTS ${mcp_connection_name}
TYPE HTTP
OPTIONS (
  host 'https://${app_host}',
  base_path '/mcp',
  client_id secret('${secret_scope_name}', '${client_id_dbs_key}'),
  client_secret secret('${secret_scope_name}', '${client_secret_dbs_key}'),
  oauth_scope 'all-apis',
  token_endpoint 'https://${workspace_host}/oidc/v1/token'
)

In [0]:
-- Idempotent — re-granting is a no-op
GRANT USE CATALOG ON CATALOG `${catalog}` TO `${workers_group_name}`

In [0]:
GRANT USE SCHEMA ON SCHEMA `${catalog}`.`${schema}` TO `${workers_group_name}`

In [0]:
-- Workers need USE CONNECTION + READ METADATA on both connections.
-- REST connection: UC functions. MCP connection: all 5 MCP services.
GRANT USE CONNECTION ON CONNECTION ${connection_name} TO `${workers_group_name}`;
GRANT READ METADATA ON CONNECTION ${connection_name} TO `${workers_group_name}`;
GRANT USE CONNECTION ON CONNECTION ${mcp_connection_name} TO `${workers_group_name}`;
GRANT READ METADATA ON CONNECTION ${mcp_connection_name} TO `${workers_group_name}`